# Meta Astryx 사용법: 코딩 에이전트에 CLI·MCP로 UI를 맡기는 오픈소스 디자인 시스템 첫 실행

위에서부터 차례로 실행해 보세요.

### Vite 프로젝트 생성과 Astryx 설치

In [ ]:
%%bash
set -eo pipefail
mkdir -p /content/app && cd /content/app
npm create vite@latest demo -- --template react-ts
cd demo && npm install
npm install @astryxdesign/core @astryxdesign/theme-neutral @stylexjs/stylex
npm install -D @astryxdesign/cli
npm ls --depth=0 | grep astryx

### CSS import, Theme, 첫 컴포넌트 작성 후 빌드

In [ ]:
%%bash
set -eo pipefail
cd /content/app/demo
cat > src/index.css <<'E'
@import '@astryxdesign/core/reset.css';
@import '@astryxdesign/core/astryx.css';
@import '@astryxdesign/theme-neutral/theme.css';
E
cat > src/App.tsx <<'E'
import {Theme} from '@astryxdesign/core/theme';
import {neutralTheme} from '@astryxdesign/theme-neutral/built';
import {Button} from '@astryxdesign/core/Button';
import {Card} from '@astryxdesign/core/Card';
import {Heading} from '@astryxdesign/core/Heading';
import {Text} from '@astryxdesign/core/Text';
import {Badge} from '@astryxdesign/core/Badge';
import {VStack} from '@astryxdesign/core/VStack';
import {HStack} from '@astryxdesign/core/HStack';
export default function App() {
  return (
    <Theme theme={neutralTheme}>
      <VStack gap={4} style={{padding: 32, maxWidth: 480}}>
        <Card>
          <VStack gap={3} style={{padding: 16}}>
            <HStack gap={2}><Heading level={2}>Hello Astryx</Heading><Badge label="Beta" /></HStack>
            <Text>Vite + React 19 first screen.</Text>
            <HStack gap={2}><Button label="Primary" /><Button label="Secondary" variant="secondary" /></HStack>
          </VStack>
        </Card>
      </VStack>
    </Theme>
  );
}
E
npx tsc -b && npx vite build 2>&1 | grep -E 'built|error'

### CLI로 컴포넌트 목록·문서·doctor

In [ ]:
%%bash
cd /content/app/demo
npx astryx --version
npx astryx component --list | head -12
npx astryx component Link | head -12
npx astryx --json component --list | jq -c '.type, (.data|keys)'
npx astryx doctor | tail -8

### manifest --json 확인

In [ ]:
%%bash
cd /content/app/demo
npx astryx manifest --json > manifest.json
jq -c '.type, .data.version, [.data.commands[].name]' manifest.json
wc -c manifest.json

### 호스팅 MCP 서버 도구 목록 조회

In [ ]:
%%bash
H='-H Content-Type:application/json -H Accept:application/json,text/event-stream'
curl -s -m 30 $H https://astryx.atmeta.com/mcp -d '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2025-03-26","capabilities":{},"clientInfo":{"name":"lab","version":"1"}}}' | sed -n 's/^data: //p'
curl -s -m 30 $H https://astryx.atmeta.com/mcp -d '{"jsonrpc":"2.0","id":2,"method":"tools/list"}' | sed -n 's/^data: //p' | jq -c '.result.tools[]|{name,input:.inputSchema.properties}'

### 에이전트가 MCP로 읽는 정보: 호스팅 서버에서 대시보드 템플릿 검색

In [ ]:
%%bash
H='-H Content-Type:application/json -H Accept:application/json,text/event-stream'
curl -s -m 30 $H https://astryx.atmeta.com/mcp -d '{"jsonrpc":"2.0","id":3,"method":"tools/call","params":{"name":"search","arguments":{"query":"dashboard"}}}' | sed -n 's/^data: //p' | jq -r '.result.content[0].text' | head -25

### 에이전트가 만든 대시보드 코드(Dashboard.tsx)와 App.tsx 저장

In [ ]:
%%bash
set -eo pipefail
cd /content/app/demo
cat > src/Dashboard.tsx <<'E'
import {Card} from '@astryxdesign/core/Card';
import {Grid} from '@astryxdesign/core/Grid';
import {VStack} from '@astryxdesign/core/Stack';
import {Table, proportional, pixel} from '@astryxdesign/core/Table';
import type {TableColumn} from '@astryxdesign/core/Table';
import {Heading, Text} from '@astryxdesign/core/Text';

interface Order extends Record<string, unknown> {
  id: string; customer: string; product: string; amount: string; status: string;
}
interface Stat { label: string; value: string; note: string; }

const stats: Stat[] = [
  {label: '총 매출', value: '₩12,480,000', note: '지난달 대비 +8.2%'},
  {label: '신규 주문', value: '342건', note: '지난달 대비 +4.5%'},
  {label: '활성 고객', value: '1,208명', note: '지난달 대비 +1.9%'},
];
const orders: Order[] = [
  {id: 'ORD-1005', customer: '김민수', product: '무선 키보드', amount: '₩89,000', status: '배송 완료'},
  {id: 'ORD-1004', customer: '이서연', product: '27인치 모니터', amount: '₩329,000', status: '배송 중'},
  {id: 'ORD-1003', customer: '박지훈', product: 'USB-C 허브', amount: '₩45,000', status: '결제 완료'},
  {id: 'ORD-1002', customer: '최유진', product: '노트북 거치대', amount: '₩32,000', status: '배송 완료'},
  {id: 'ORD-1001', customer: '정하늘', product: '웹캠', amount: '₩67,000', status: '취소'},
];
const columns: TableColumn<Order>[] = [
  {key: 'id', header: '주문번호', width: pixel(120)},
  {key: 'customer', header: '고객', width: proportional(1)},
  {key: 'product', header: '상품', width: proportional(2)},
  {key: 'amount', header: '금액', width: proportional(1)},
  {key: 'status', header: '상태', width: proportional(1)},
];

export default function Dashboard() {
  return (
    <VStack gap={6} padding={6}>
      <VStack gap={1}>
        <Heading level={1}>대시보드</Heading>
        <Text type="supporting">이번 달 주요 지표와 최근 주문을 확인하세요.</Text>
      </VStack>
      <Grid columns={{minWidth: 200, max: 3}} gap={4}>
        {stats.map(stat => (
          <Card key={stat.label}>
            <VStack gap={2}>
              <Text type="label" color="secondary" display="block">{stat.label}</Text>
              <Heading level={2}>{stat.value}</Heading>
              <Text type="supporting" display="block">{stat.note}</Text>
            </VStack>
          </Card>
        ))}
      </Grid>
      <VStack gap={3}>
        <Heading level={2}>최근 주문</Heading>
        <Table data={orders} columns={columns} idKey="id" />
      </VStack>
    </VStack>
  );
}
E
cat > src/App.tsx <<'E'
import {Theme} from '@astryxdesign/core/theme';
import {neutralTheme} from '@astryxdesign/theme-neutral/built';
import Dashboard from './Dashboard';

export default function App() {
  return (
    <Theme theme={neutralTheme}>
      <Dashboard />
    </Theme>
  );
}
E
ls src

### 대시보드 타입 검사·빌드, Astryx doctor로 점검

In [ ]:
%%bash
set -eo pipefail
cd /content/app/demo
npx tsc -b && echo TSC_OK
npx vite build 2>&1 | grep -E "built|error"
npx astryx doctor --built dist 2>&1 | tail -15
echo "raw div 개수: $(grep -c '<div' src/Dashboard.tsx || true)"
grep -E "^import" src/Dashboard.tsx